# 13. When SymPy fails, and what to do about it

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ktwyw/sympy-for-engineers/blob/main/notebooks/13_when_sympy_fails.ipynb)

The earlier notebooks show SymPy succeeding. This one collects the ways it goes wrong, because
knowing the limits of a tool is part of knowing the tool. Every case below is one that students
actually hit, with the reason and the fix.

| # | Symptom | Section |
| --- | --- | --- |
| 1 | ugly decimals such as `0.333333333333333` in a result | 13.1 |
| 2 | two equal expressions compare as `False` | 13.2 |
| 3 | an obvious simplification does not happen | 13.3, 13.4 |
| 4 | `integrate` returns the integral unchanged | 13.5 |
| 5 | `solve` raises `NotImplementedError`, or `nsolve` fails | 13.6 |
| 6 | `dsolve` or `pdsolve` cannot solve the equation | 13.7 |
| 7 | a solution is returned that makes no physical sense | 13.8 |
| 8 | the calculation never finishes | 13.9 |
| 9 | the formula is right but the numbers are wrong | 13.10 |

In [1]:
import time

import numpy as np
import sympy as sp
from scipy.integrate import quad

sp.init_printing(use_latex="mathjax")
x, a, b = sp.symbols("x a b")

## 13.1 Floats creep in

Python evaluates `1/3` *before* SymPy sees it, and a float can never be turned back into the
exact number you meant.

In [2]:
x ** (1 / 3), x ** sp.Rational(1, 3)

⎛ 0.333333333333333  3 ___⎞
⎝x                 , ╲╱ x ⎠

In [3]:
sp.Rational(0.1), sp.Rational("0.1"), sp.Rational(1, 10)

⎛3602879701896397             ⎞
⎜─────────────────, 1/10, 1/10⎟
⎝36028797018963968            ⎠

The first result is the binary fraction your computer actually stores for `0.1`.

**Fix.** Write exact numbers as `sp.Rational(1, 3)`, `sp.S(1)/3` or `sp.sqrt(x)`. Keep decimal
data (measurements, property values) out of the symbolic work: derive with symbols and
substitute the numbers at the end. If floats are already there, `sp.nsimplify` repairs them.

In [4]:
messy = sp.integrate(x**0.5, x)
messy, sp.nsimplify(messy)

⎛                           3/2⎞
⎜                   1.5  2⋅x   ⎟
⎜0.666666666666667⋅x   , ──────⎟
⎝                          3   ⎠

## 13.2 `==` does not test mathematical equality

`==` asks "are these the same expression tree?", not "are these equal for all $x$?".

In [5]:
left = (x + 1) ** 2
right = x**2 + 2 * x + 1
left == right

False

**Fix.** Simplify the difference and compare with zero, or use `.equals`. This is the pattern
used for every verification in these notebooks.

In [6]:
assert sp.simplify(left - right) == 0
left.equals(right)

True

## 13.3 Missing assumptions

SymPy assumes a symbol could be any complex number. Many familiar rules are false in that case,
so it refuses to apply them.

In [7]:
p, q = sp.symbols("p q", positive=True)

generic = (sp.sqrt(x**2), sp.expand_log(sp.log(a * b)), sp.integrate(sp.exp(-a * x), (x, 0, sp.oo)))
informed = (sp.sqrt(p**2), sp.expand_log(sp.log(p * q)), sp.integrate(sp.exp(-p * q), (q, 0, sp.oo)))
display(generic, informed)

⎛                   ⎧    1                      π⎞
⎜                   ⎪    ─       for │arg(a)│ < ─⎟
⎜                   ⎪    a                      2⎟
⎜   ____            ⎪                            ⎟
⎜  ╱  2             ⎪∞                           ⎟
⎜╲╱  x  , log(a⋅b), ⎨⌠                           ⎟
⎜                   ⎪⎮  -a⋅x                     ⎟
⎜                   ⎪⎮ ℯ     dx     otherwise    ⎟
⎜                   ⎪⌡                           ⎟
⎜                   ⎪0                           ⎟
⎝                   ⎩                            ⎠

⎛                    1⎞
⎜p, log(p) + log(q), ─⎟
⎝                    p⎠

The integral with a generic `a` comes back as a `Piecewise`: "it equals $1/a$ **if** the real
part of $a$ is positive, otherwise it does not converge". That is correct, just not what you
wanted.

**Fix.** Declare what you know: `positive=True`, `real=True`, `integer=True`.

A related trap: two symbols with the same name but different assumptions are **different
symbols**, and they will not cancel.

In [8]:
x_plain = sp.Symbol("x")
x_positive = sp.Symbol("x", positive=True)
x_plain == x_positive, x_plain - x_positive      # prints as "x - x", and is not zero

(False, -x + x)

**Fix.** Define each symbol once, at the top of the notebook, and reuse it.

Assumptions also change what `solve` returns:

In [9]:
sp.solve(x_plain**2 - 4, x_plain), sp.solve(x_positive**2 - 4, x_positive)

([-2, 2], [2])

## 13.4 `simplify` is a guess, not a guarantee

There is no single "simplest form". `simplify` tries a set of strategies and keeps the shortest
result, which may not be the form you need. When it does not deliver, use the specific tool.

| You want | Use |
| --- | --- |
| multiply out | `expand` |
| a product of factors | `factor` |
| cancel a common factor in a fraction | `cancel` |
| group by powers of a variable | `collect` |
| trigonometric identities | `trigsimp` |
| no square roots in the denominator | `radsimp` |
| another family of functions | `.rewrite(sp.exp)`, `.rewrite(sp.cos)`, … |

In [10]:
surd = 1 / (sp.sqrt(2) + sp.sqrt(3))
sp.simplify(surd), sp.radsimp(surd)

⎛   1             ⎞
⎜───────, -√2 + √3⎟
⎝√2 + √3          ⎠

In [11]:
# rewrite converts between families of functions: hyperbolic to exponential, and Euler's formula
sp.tanh(x).rewrite(sp.exp), sp.exp(sp.I * x).rewrite(sp.cos)

⎛ x    -x                   ⎞
⎜ℯ  - ℯ                     ⎟
⎜────────, ⅈ⋅sin(x) + cos(x)⎟
⎜ x    -x                   ⎟
⎝ℯ  + ℯ                     ⎠

## 13.5 Integrals with no closed form

Most integrals cannot be written with elementary functions. SymPy then returns the integral
unevaluated. This is an answer ("I cannot do this"), not an error, so check for it.

In [12]:
stuck = sp.integrate(sp.sin(sp.sin(x)), (x, 0, 1))
stuck, isinstance(stuck, sp.Integral)

(Integral(sin(sin(x)), (x, 0, 1)), True)

**Fix.** If you need a number, integrate numerically: `.evalf()` on the unevaluated integral, or
`scipy.integrate.quad` on a lambdified integrand.

In [13]:
value_sympy = stuck.evalf()
value_scipy, _ = quad(sp.lambdify(x, sp.sin(sp.sin(x))), 0, 1)
assert abs(value_sympy - value_scipy) < 1e-12
value_sympy, value_scipy

(0.430606103120691, 0.4306061031206906)

Sometimes a closed form exists in terms of a **special function**. That is a real answer:
such functions can be evaluated, differentiated and plotted like any other.

In [14]:
sp.integrate(sp.exp(-x**2), (x, 0, 1))        # the error function

√π⋅erf(1)
─────────
    2    

## 13.6 Equations with no closed form

`solve` looks for formulas. For a transcendental equation there usually is none.

In [15]:
try:
    sp.solve(sp.Eq(x, sp.cos(x)), x)
except NotImplementedError as err:
    print("NotImplementedError:", str(err).splitlines()[0])

NotImplementedError: multiple generators [x, cos(x)]


In [16]:
sp.nsolve(sp.Eq(x, sp.cos(x)), x, 1)          # numerical root, starting from the guess x = 1

0.739085133215161

`nsolve` has its own failure mode: it needs a starting guess close enough to the root. The
Colebrook equation of notebook 2, written directly in terms of the friction factor $f$, defeats
it from guesses that look perfectly reasonable. The default iteration takes a first step far
outside the physical range, into negative $f$ where the square root is not real.

In [17]:
f = sp.symbols("f", positive=True)
colebrook = sp.Eq(1 / sp.sqrt(f), -2 * sp.log(1e-4 / 3.7 + 2.51 / (1e5 * sp.sqrt(f)), 10))

for guess in (0.01, 0.02, 0.05):
    try:
        print(guess, sp.nsolve(colebrook, f, guess))
    except ValueError:
        print(f"guess {guess}: nsolve failed")

guess 0.01: nsolve failed
guess 0.02: nsolve failed
guess 0.05: nsolve failed


**Fixes**, in order of preference:

1. **Reformulate.** Solve for $X = 1/\sqrt f$, in which the equation is nearly linear
   (done in notebook 2).
2. **Bracket the root.** Give an interval that contains it and use bisection, which cannot escape.
3. Plot the function first, and read a good guess from the graph.

In [18]:
root = sp.nsolve(colebrook, f, (0.005, 0.1), solver="bisect")
assert abs(root - 0.018514) < 1e-5
root

0.0185138660774716

## 13.7 Differential equations

`dsolve` handles linear equations and a catalogue of special nonlinear ones. Typical nonlinear
engineering models are outside it. The Van der Pol oscillator:

In [19]:
t = sp.symbols("t")
y = sp.Function("y")
van_der_pol = y(t).diff(t, 2) - (1 - y(t) ** 2) * y(t).diff(t) + y(t)

try:
    sp.dsolve(van_der_pol, y(t))
except NotImplementedError:
    print("NotImplementedError: dsolve has no method for this equation")

NotImplementedError: dsolve has no method for this equation


**Fix.** Integrate numerically with `scipy.integrate.solve_ivp`, using `lambdify` for the
right-hand side (notebooks 8 and 12).

Partial differential equations are harder still. `pdsolve` covers only first-order equations,
so the heat equation is already out of reach:

In [20]:
u = sp.Function("u")
alpha, k = sp.symbols("alpha k", positive=True)
heat = u(x, t).diff(t) - alpha * u(x, t).diff(x, 2)

try:
    sp.pdsolve(heat)
except NotImplementedError:
    print("NotImplementedError: pdsolve cannot solve the heat equation")

NotImplementedError: pdsolve cannot solve the heat equation


**What still works.** SymPy can *check* a solution you found by separation of variables, or one
from a textbook. Substitute it and simplify the residual to zero.

In [21]:
trial = sp.exp(-alpha * k**2 * t) * sp.sin(k * x)
residual = heat.subs(u(x, t), trial).doit()
assert sp.simplify(residual) == 0

Notebook 14 shows how to solve PDEs numerically, with SymPy designing and checking the method:
discretize in space, pass the resulting ODEs to `solve_ivp`, and verify the order of accuracy.

## 13.8 Answers that are mathematically right and physically wrong

`solve` returns every solution of the algebra and knows nothing about physics. Three patterns:

- **Extra roots.** A cubic equation of state has three volume roots; one is never physical
  (notebook 2). A steady state can have a negative concentration (notebook 9).
- **Generic solutions.** Special cases are silently dropped:

In [22]:
sp.solve(a * x - b, x)          # true unless a = 0, which is not mentioned

⎡b⎤
⎢─⎥
⎣a⎦

- **Formulas outside their range.** Stokes' law in notebook 1 gives a number for any particle
  size; only the Reynolds number check told us whether to believe it.

**Fix.** Check signs, ranges, limiting cases and units every time. Declaring symbols `positive`
removes some unphysical roots automatically.

## 13.9 Expression swell and long run times

Symbolic results can grow explosively. The determinant of a general $n \times n$ matrix has
$n!$ terms:

In [23]:
for n in (2, 3, 4, 5):
    general = sp.Matrix(n, n, lambda i, j: sp.Symbol(f"m{i}{j}"))
    start = time.perf_counter()
    operations = sp.count_ops(general.det())
    print(f"{n} x {n}: {operations:5d} operations   {time.perf_counter() - start:6.2f} s")

2 x 2:     3 operations     0.00 s
3 x 3:    17 operations     0.00 s
4 x 4:    95 operations     0.13 s


5 x 5:   599 operations     3.24 s


The $6 \times 6$ case took more than two minutes on the machine that built this notebook, and a
$10 \times 10$ symbolic determinant is hopeless, while NumPy handles a $1000 \times 1000$
numerical one instantly.

**Fixes.**

- Substitute numbers for parameters you do not need symbolically, *before* the heavy step.
- Exploit structure: sparse, triangular or block matrices stay small.
- Ask whether you need the formula at all. Often only the numbers are needed.
- If a call runs for more than a minute, interrupt it and rethink the approach.
  Long waits rarely end in a useful result.

## 13.10 The formula is right, the number is wrong

Converting an exact formula to floating point can destroy its accuracy. The expression
$(1 - \cos x)/x^2$ tends to $1/2$ as $x \to 0$:

In [24]:
expr = (1 - sp.cos(x)) / x**2
sp.limit(expr, x, 0), sp.series(expr, x, 0, 4)

⎛          2        ⎞
⎜     1   x     ⎛ 4⎞⎟
⎜1/2, ─ - ── + O⎝x ⎠⎟
⎝     2   24        ⎠

In [25]:
naive = sp.lambdify(x, expr, "numpy")
small = np.array([1e-1, 1e-4, 1e-8])
naive(small)

array([0.49958347, 0.5       , 0.        ])

At $x = 10^{-8}$ the float result is 0 instead of 0.5. In double precision $\cos(10^{-8})$
rounds to exactly 1, so the subtraction loses every significant digit. This is catastrophic
cancellation, and it appears whenever nearly equal numbers are subtracted.

**Fixes.** Rewrite the formula so the subtraction disappears, use the series near the trouble
spot, or evaluate with extra digits.

In [26]:
stable_form = 2 * sp.sin(x / 2) ** 2 / x**2          # identical, by the half-angle formula
assert sp.simplify(stable_form - expr) == 0

stable = sp.lambdify(x, stable_form, "numpy")
assert np.allclose(stable(small), 0.5, atol=1e-3)
stable(small), expr.subs(x, sp.Rational(1, 10**8)).evalf(20)

(array([0.49958347, 0.5       , 0.5       ]), 0.49999999999999999583)

Two smaller `lambdify` surprises:

- An expression that turns out to be **constant** gives a function that returns a single
  number, not an array matching the input. Code that plots it will then fail.
- `lambdify` does not have a translation for every special function (notebook 8 supplied one
  for the elliptic integral).

In [27]:
slope = sp.lambdify(x, sp.diff(3 * x + 1, x), "numpy")       # the derivative is the constant 3
grid = np.linspace(0, 1, 5)
slope(grid), np.full_like(grid, slope(grid))                 # second form is safe for plotting

(3, array([3., 3., 3., 3., 3.]))

## A decision guide

| The task | Symbolic (SymPy) | Numerical (NumPy, SciPy) |
| --- | --- | --- |
| derive or check a formula | yes | no |
| derivatives, Jacobians, sensitivities | yes | approximate only |
| limits, series, special cases | yes | no |
| small linear systems with parameters | yes | no |
| large linear systems | no | yes |
| nonlinear equations | if a closed form exists | always works with a good guess |
| integrals | if a closed form exists | always |
| ODEs | linear and textbook cases | always |
| PDEs | checking solutions only | yes, with a suitable package |
| fitting data, statistics | no | yes |

The two are partners. The most effective workflow is the one from notebook 8: derive
symbolically, verify, then compute numerically.

## A debugging checklist

When SymPy does something unexpected, ask in this order:

1. Is there a float where an exact number should be? (13.1)
2. Did I declare the assumptions I know? (13.3)
3. Am I comparing with `==` instead of simplifying a difference? (13.2)
4. Did the function return the problem unsolved (an `Integral`, an empty list, a `Piecewise`)? (13.5)
5. Is there a specific tool better than `simplify` for the form I want? (13.4)
6. Does a closed form exist at all? If not, go numerical. (13.6, 13.7)
7. Have I checked the result against a limiting case, a number, or the units? (13.8)

## Exercises

1. Explain why `sp.solve(x**2 - 2.0, x)` and `sp.solve(x**2 - 2, x)` print differently.
2. `sp.simplify(sp.sqrt(x**2) - x)` is not zero. Find two different ways to make it zero.
3. The integral $\int_0^1 x^x\,dx$ has no closed form. Evaluate it numerically in two ways and
   compare with the series $\sum_{n=1}^{\infty} (-1)^{n+1} n^{-n}$. Hint: build the integral
   with `sp.Integral(...)` before calling `.evalf()`; the object returned by `integrate` does
   not evaluate here.
4. The function $\dfrac{e^x - 1}{x}$ suffers from cancellation near zero. Show the problem with
   `lambdify`, then fix it. (NumPy has `expm1` for exactly this purpose.)
5. Find the two positive roots of $\tan x = x$ below 10 (besides the obvious root $x = 0$).
   Why does `nsolve` need care here, and how does bracketing help?